# Train your personal voice model

Run this notebook in Google Colab with a CUDA GPU runtime. It uses the repo's reviewed-dataset pipeline and official Qwen3-TTS trainer. The trained checkpoint is saved to Google Drive and can optionally be uploaded to a private Hugging Face model repository.

**Data requirement:** the current ~45-second example is not enough for a strong model. For a practical starting point, upload around 10–30 minutes of clean, varied speech recorded by you. This is a recommendation, not an official hard minimum. Do not use recordings of other people without permission.

**Important:** Colab GPU availability and session time are not guaranteed. Save outputs to Drive; don't store model weights or private recordings in this public GitHub repo.

In [ ]:
import os, sys, subprocess
from pathlib import Path
import torch
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('In Colab, choose Runtime > Change runtime type > GPU, then rerun this cell.')
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
REPO = Path('/content/Voice-Clone')
if not (REPO / '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/harshyadavv2456/Voice-Clone.git', str(REPO)], check=True)
os.chdir(REPO)
DATA = Path('/content/drive/MyDrive/VoiceCloneDataset')
RAW = DATA / 'raw'
PREPARED = DATA / 'prepared'
TRAINING = DATA / 'training'
OUTPUT = DATA / 'model-output' / 'harsh-yadav-qwen3'
RAW.mkdir(parents=True, exist_ok=True)
TRAINING.mkdir(parents=True, exist_ok=True)
print('Put your own speech recordings in:', RAW)
print('Expected audio formats: WAV, MP3, M4A, FLAC. Keep them in Drive, not in GitHub.')

In [ ]:
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg', 'sox', 'libsox-fmt-all'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print('Audio and Qwen dependencies installed.')

## Step 1 — create and review speech segments

Run the next cell once. It writes prepared WAV clips and a CSV manifest to your Drive folder. Transcripts are machine-generated suggestions. Open VoiceCloneDataset/prepared/manifest.csv from Google Drive, correct each transcript to exactly match what you said, and set approved to YES only for clean clips with correct text. Leave all other rows as NO. Do not change the audio or ref_audio file paths.

In [ ]:
if not any(p.is_file() for p in RAW.rglob('*') if p.suffix.lower() in {'.wav', '.mp3', '.m4a', '.flac', '.ogg', '.aac', '.mp4', '.webm'}):
    raise RuntimeError('No audio files found. Upload your recordings to MyDrive/VoiceCloneDataset/raw and rerun this cell.')
subprocess.run([sys.executable, 'training/prepare_dataset.py', '--input-dir', str(RAW), '--output-dir', str(PREPARED), '--whisper-model', 'base', '--language', 'en'], check=True)
print('Manifest to review:', PREPARED / 'manifest.csv')
print('PAUSE HERE: review/correct the CSV in Drive before running the training cell below.')

## Step 2 — fine-tune after transcript review

Only run this cell after you have reviewed the manifest in Drive and marked clean, accurate rows approved=YES. It builds the official JSONL, prepares audio tokens, and fine-tunes the 0.6B Base checkpoint. Defaults are batch size 2 and 3 epochs; a GPU out-of-memory error may require a lower batch size or a larger GPU. The checkpoint is written to Drive so it survives a Colab disconnect.

In [ ]:
subprocess.run([sys.executable, 'training/build_train_jsonl.py', '--manifest', str(PREPARED / 'manifest.csv'), '--output', str(TRAINING / 'train_raw.jsonl')], check=True)
os.environ['RAW_JSONL'] = str(TRAINING / 'train_raw.jsonl')
os.environ['TRAIN_JSONL'] = str(TRAINING / 'train_with_codes.jsonl')
os.environ['OUTPUT_DIR'] = str(OUTPUT)
os.environ['MODEL_ID'] = 'Qwen/Qwen3-TTS-12Hz-0.6B-Base'
os.environ['TOKENIZER_ID'] = 'Qwen/Qwen3-TTS-Tokenizer-12Hz'
os.environ['DEVICE'] = 'cuda:0'
os.environ['BATCH_SIZE'] = '2'
os.environ['LEARNING_RATE'] = '2e-5'
os.environ['EPOCHS'] = '3'
os.environ['SPEAKER_NAME'] = 'harsh_yadav'
subprocess.run(['bash', 'training/run_finetuning.sh'], check=True)
CHECKPOINT = OUTPUT / 'checkpoint-epoch-2'
print('Candidate trained checkpoint:', CHECKPOINT)

## Step 3 — optionally publish the complete model privately

This creates a private Hugging Face model repository. You need a Hugging Face account and a write token. Don't paste the token into a notebook cell or save it in Drive. The prompt below is hidden. Set your desired repo id, for example YOUR_USERNAME/harsh-yadav-qwen3-voice. Keep the repository private because the model can reproduce your voice.

In [ ]:
from getpass import getpass
from huggingface_hub import HfApi
HF_TOKEN = getpass('Hugging Face write token (hidden): ')
HF_REPO_ID = input('Private model repo id (username/harsh-yadav-qwen3-voice): ').strip()
if '/' not in HF_REPO_ID:
    raise ValueError('Use the format username/model-name')
os.environ['HF_TOKEN'] = HF_TOKEN
subprocess.run([sys.executable, 'training/publish_model.py', '--model-dir', str(CHECKPOINT), '--repo-id', HF_REPO_ID], check=True)
print('Private model upload complete. Download it on your Mac with training/download_model.py.')